# Phase 1: Problem Framing And Dataset Understanding

This notebook covers the first phase of the graph-based recommendation system project.

Our goal in this phase is not to build a model yet. Our goal is to understand the recommendation problem, inspect the dataset, and justify why this problem should be represented as a graph.


## 1. Concept

A recommendation system tries to estimate which items are most relevant for a user.

Examples:

- recommend movies to viewers,
- recommend products to customers,
- recommend songs to listeners,
- recommend articles to readers.

In this project, we will start with a user-item interaction dataset and frame the problem as a ranking problem.


## 2. Why We Apply This

We study the problem carefully before building a model because the quality of a recommendation project depends on the clarity of the problem definition.

Before modeling, we need to answer:

- What is a user?
- What is an item?
- What counts as an interaction?
- Are we solving rating prediction or ranking?
- What fields exist in the dataset?
- Why is this a graph problem instead of only a table problem?


## 3. Math Behind The Problem

A standard way to represent recommendation data is the interaction matrix:

$$R \in \mathbb{R}^{m \times n}$$

Where:

- $m$ is the number of users,
- $n$ is the number of items,
- $R_{ui}$ describes whether user $u$ interacted with item $i$.

For implicit feedback:

- $R_{ui} = 1$ means an observed interaction,
- $R_{ui} = 0$ means unobserved, not necessarily negative.

This same data can be viewed as a bipartite graph:

$$A = \begin{bmatrix} 0 & R \\ R^T & 0 \end{bmatrix}$$

This means:

- users form one node set,
- items form the other node set,
- interactions become edges between users and items.


## 4. Dataset Choice For The First Version

For the first version of this project, the recommended dataset is **MovieLens 100K**.

Why this is a good starter dataset:

- it is widely used,
- it is small enough for quick experiments,
- it has clear user-item interactions,
- it is easy to explain in an academic report,
- it is strong enough for baseline and graph-based recommendation experiments.

Expected raw columns:

- `user_id`
- `item_id`
- `rating`
- `timestamp`

Even if ratings exist, we can convert the first version into an implicit-feedback recommendation task.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("data/raw/recommendation/movielens-100k/u.data")
DATA_PATH

## 5. Load The Dataset

The original `u.data` file in MovieLens 100K is tab-separated and usually has no header row.

We will load it with the expected schema:

- `user_id`
- `item_id`
- `rating`
- `timestamp`

If the file is not present yet, the cell below will remind us where it should go.


In [ ]:
if not DATA_PATH.exists():
    print("Dataset not found yet.")
    print("Place MovieLens 100K at: data/raw/recommendation/movielens-100k/u.data")
else:
    df = pd.read_csv(
        DATA_PATH,
        sep='\t',
        names=['user_id', 'item_id', 'rating', 'timestamp']
    )
    print(df.head())
    print(df.shape)


## 6. Inspect The Schema

In this section, we verify what fields exist and what they mean.

This matters because our later graph design depends on it:

- `user_id` will define user nodes,
- `item_id` will define item nodes,
- `rating` may be used as a threshold or edge weight,
- `timestamp` may later support temporal train/test splits.


In [ ]:
if DATA_PATH.exists():
    print(df.info())
    display(df.head())
    display(df.describe(include='all'))


## 7. Basic Dataset Statistics

Before building a recommender, we want the first summary statistics.

The key quantities are:

- number of users,
- number of items,
- number of interactions,
- average interactions per user,
- average interactions per item,
- sparsity of the interaction matrix.

These numbers help us understand whether the recommendation problem is dense or sparse, and sparsity is especially important because real recommendation datasets are usually very sparse.


In [ ]:
if DATA_PATH.exists():
    num_users = df['user_id'].nunique()
    num_items = df['item_id'].nunique()
    num_interactions = len(df)
    density = num_interactions / (num_users * num_items)
    sparsity = 1 - density

    stats = {
        'num_users': num_users,
        'num_items': num_items,
        'num_interactions': num_interactions,
        'avg_interactions_per_user': num_interactions / num_users,
        'avg_interactions_per_item': num_interactions / num_items,
        'density': density,
        'sparsity': sparsity,
    }

    for key, value in stats.items():
        print(f"{key}: {value}")


## 8. Why This Is A Graph Problem

This recommendation task is a graph problem because the important signal is relational.

We care about patterns such as:

- users who interact with similar items,
- items that attract similar groups of users,
- higher-order similarity through shared neighbors.

Example intuition:

- User A watched Item 1 and Item 2.
- User B watched Item 2 and Item 3.
- The shared connection through Item 2 suggests User A may also be interested in Item 3.

This kind of multi-hop relational signal is one of the main reasons graph-based recommendation works well.


## 9. Phase 1 Decision Log

By the end of this phase, we want to be able to answer these questions clearly.

- What dataset are we using?
- What is the prediction task?
- Are we using explicit ratings or implicit interactions first?
- What are the important dataset fields?
- Why does a bipartite graph make sense here?

Current intended answers for the first version:

- Dataset: MovieLens 100K
- First task: top-K item recommendation
- First learning setting: implicit-feedback ranking
- Graph type: bipartite user-item graph


## 10. Next Step

After Phase 1, the next phase is data preparation and bipartite graph construction.

That means we will next:

- map raw user IDs to contiguous indices,
- map raw item IDs to contiguous indices,
- create train/validation/test splits,
- build the first bipartite graph from the training interactions.
